# 범용 손그림 탐지 모델 학습 (Stage A) — Quick Draw + HTP person

**산출물 1: 범용 손그림(line-art) 객체탐지 모델.** Quick Draw 스케치를 랜덤 배치한
합성 장면으로 YOLO를 학습해 `htp_yolo/<SKETCH_MODEL_NAME>` 을 만든다.

### person 클래스는 HTP에서 온다 (S15P11B209-685)
**Quick Draw에는 '사람 전체' 카테고리가 없다.** `face`·`arm`·`leg` 같은 부위만 있어서,
카테고리를 100종으로 늘려도 person은 생기지 않았다(v2까지의 한계). 그래서 HTP 데이터셋의
`사람전체` bbox를 크롭해 Quick Draw 스케치와 **같은 방식으로 합성 캔버스에 붙인다.**
남자사람·여자사람을 합쳐 단일 `person` 클래스로 쓴다.

이때 **하위 라벨(눈·팔·손·다리·발·얼굴…)을 함께 옮긴다.** `사람전체` bbox 안에는
하위 객체가 평균 22개 들어 있어서, person 박스만 달면 그것들이 전부 '라벨 없는 배경'이
되어 기존 `eye`·`arm`·`hand` 클래스를 망가뜨린다. 자세한 이유는 Config 셀 주석 참고.

> HTP **원본 이미지를 데이터셋으로 섞지는 않는다.** 크롭한 스프라이트만 쓴다 —
> 원본에는 태양·구름·잔디가 함께 그려져 있어 통째로 넣으면 라벨 안 된 객체가
> negative가 되어 나머지 100클래스를 오염시킨다.

### HTP 없이도 단독으로 돌아간다
HTP를 못 찾으면 **경고만 내고 person 없이 100클래스로** 끝까지 실행된다.
HTP 파인튜닝은 [htp_finetune.ipynb](htp_finetune.ipynb)가 담당하며, 이 노트북의 산출물
`.pt` 파일 하나만 입력으로 받는다. 즉 산출물 의존은 여전히 **파일 경유 단방향**이다.

```
             (선택) HTP 사람 스프라이트
                       │
                       ▼
sketch_base.ipynb ──> sketch_base*.pt ──> htp_finetune.ipynb ──> htp_best*.pt
```

### 이 노트북이 절대 건드리지 않는 것
`htp_best.pt`, `stageB_htp/`, `classes.json`, **HTP 원본 데이터**(읽기 전용).
클래스를 바꿔가며 몇 번을 재실행해도 HTP 산출물은 안전하다.
(분리 전 단일 노트북에서는 Run All이 `htp_best.pt`를 덮어썼다.)

### 클래스를 확장하려면
`Config`의 `QD_CATEGORIES`에 카테고리를 추가하고 `QD_TRAIN_IMAGES`를 함께 올린다.
클래스만 늘리고 이미지 수를 그대로 두면 희귀 클래스가 학습되지 않는다.
**그리고 `SKETCH_MODEL_NAME`을 새 이름으로 바꿔** 기존 가중치를 보존한다 —
`htp_best.pt`는 기존 `sketch_base.pt`에서 파생된 산출물이라 그 조상을 지우면 계보가 끊긴다.

### ⚠️ 캐시 무효화 규칙
| 바꾼 것 | 지워야 반영되는 것 |
|---|---|
| `QD_CATEGORIES`, `HTP_SPRITE_ENABLE` 등 **클래스 목록**이 바뀌는 변경 | 자동 감지 — `data.yaml`의 클래스와 비교해 다르면 다시 만든다 |
| `QD_TRAIN_IMAGES`, `QD_CANVAS`, `HTP_PERSON_PROB` 등 합성 파라미터 | `htp_yolo/stageA_quickdraw/` 를 직접 삭제 |
| 학습 파라미터(`QD_EPOCHS`, `MODEL_ARCH` …) | 없음 — `FORCE_RETRAIN=True`로 재학습 |

Config를 고친 뒤에는 **커널 재시작 → Run All**.

## 1. 환경 세팅

In [ ]:
# 필요 패키지 설치 (JupyterHub/VSCode 공통). 이미 설치돼 있으면 그대로 통과.
%pip install -q "ultralytics>=8.3.0" opencv-python-headless pillow pyyaml tqdm matplotlib pandas

In [ ]:
# Stage A는 zip·HTP 관련 의존이 없다 — Quick Draw 다운로드와 합성만 필요하다.
import os

# ── GPU 선택: 물리 GPU 6번만 사용 (공용 서버 지정) ────────────────
#   ⚠️ import torch 보다 반드시 먼저 실행돼야 적용된다. 이미 torch가 로드된
#      뒤면 무시되므로 커널 재시작 후 이 셀부터 다시 실행할 것.
#   ⚠️ 값은 문자열이어야 한다. CUDA_VISIBLE_DEVICES="6" 을 걸면 torch에는
#      그 GPU가 유일한 장치(cuda:0)로 보이므로 Config의 DEVICE 는 6이 아니라
#      0 으로 두어야 한다(그대로 두면 됨).
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import sys, json, random, shutil, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    TORCH_VER = torch.__version__
except ModuleNotFoundError:
    torch = None; CUDA_AVAILABLE = False; TORCH_VER = "(미설치)"

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("PyTorch     :", TORCH_VER)
print("CUDA 사용가능:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("GPU         :", torch.cuda.get_device_name(0), f"(x{torch.cuda.device_count()})")

In [ ]:
import io, urllib.request
from urllib.parse import quote
from PIL import Image, ImageDraw, ImageChops

In [ ]:
# ── 한글 폰트 설정 (matplotlib 라벨 깨짐 방지) ──────────────────────
# Windows/Linux/macOS 공통. 이 셀을 먼저 실행하면 이후 모든 시각화에 한글이 적용된다.
# JupyterHub(리눅스, sudo 없음)처럼 시스템에 한글 폰트가 없으면 NanumGothic을
# 유저 폴더에 자동 내려받아 등록한다 — apt/root 권한이 필요 없다.
import urllib.request
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 유저 홈에 폰트를 캐시(쓰기 가능·재실행 시 재사용). 실패하면 현재 폴더로 대체.
_FONT_URL = ("https://github.com/google/fonts/raw/main/ofl/nanumgothic/"
             "NanumGothic-Regular.ttf")
try:
    _FONT_CACHE = Path.home() / ".fonts"
    _FONT_CACHE.mkdir(parents=True, exist_ok=True)
except Exception:
    _FONT_CACHE = Path("fonts"); _FONT_CACHE.mkdir(exist_ok=True)
_FONT_DL = _FONT_CACHE / "NanumGothic-Regular.ttf"


def _download_korean_font():
    # 이미 받아둔 게 있으면 재사용, 없으면 내려받는다. 반환: 성공 시 파일 경로.
    if _FONT_DL.exists() and _FONT_DL.stat().st_size > 100_000:
        return _FONT_DL
    try:
        req = urllib.request.Request(_FONT_URL, headers={"User-Agent": "htp/1.0"})
        with urllib.request.urlopen(req, timeout=120) as r:
            _FONT_DL.write_bytes(r.read())
        print(f"NanumGothic 다운로드: {_FONT_DL}")
        return _FONT_DL
    except Exception as e:
        print(f"⚠️ 폰트 다운로드 실패({type(e).__name__}: {e}) — 네트워크/프록시를 확인하세요.")
        return None


def _apply_font(fp):
    font_manager.fontManager.addfont(str(fp))
    name = font_manager.FontProperties(fname=str(fp)).get_name()
    plt.rcParams["font.family"] = name
    plt.rcParams["axes.unicode_minus"] = False
    return name


def setup_korean_font():
    candidates = [
        r"C:\Windows\Fonts\malgun.ttf",        # Windows 맑은 고딕
        r"C:\Windows\Fonts\NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",          # Linux (apt)
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",               # macOS
        "/Library/Fonts/AppleGothic.ttf",
        str(_FONT_DL),                          # 이전 실행에서 받아둔 것
    ]
    for fp in candidates:
        if Path(fp).exists():
            name = _apply_font(fp)
            print(f"한글 폰트: {name}  ({fp})")
            return fp, name
    # 설치된 폰트 이름으로 시도(파일 경로로 못 찾은 경우)
    for name in ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]:
        if any(f.name == name for f in font_manager.fontManager.ttflist):
            plt.rcParams["font.family"] = name
            plt.rcParams["axes.unicode_minus"] = False
            print(f"한글 폰트(이름): {name}")
            return None, name
    # 마지막 수단: 유저 폴더로 다운로드해 등록 (JupyterHub 등 root 없는 환경)
    fp = _download_korean_font()
    if fp is not None:
        name = _apply_font(fp)
        print(f"한글 폰트(다운로드): {name}  ({fp})")
        return str(fp), name
    print("⚠️ 한글 폰트를 찾지 못했습니다. 라벨이 깨질 수 있습니다.")
    return None, None

KOREAN_FONT_PATH, KOREAN_FONT_NAME = setup_korean_font()

## 2. Config  ⚙️

**환경을 바꿀 때는 이 셀만 수정한다.** 기본값은 **본학습용**이다
(카테고리당 15,000장 · 합성 40,000장 · 40 epoch — v2·v3를 학습한 설정과 동일).

빠르게 돌려보려면 `QD_PER_CATEGORY=800`, `QD_TRAIN_IMAGES=1500`, `QD_EPOCHS=10`으로
낮춘다. 단 그 값으로 만든 가중치는 세대 비교에 쓸 수 없다.

In [ ]:
# ═══════════════════════ Config (Stage A) ═══════════════════════
# ⚠️ 절대경로 필수: 상대경로를 쓰면 Ultralytics가 project= 인자를
#    ~/AppData/Roaming/Ultralytics/settings.json 의 runs_dir(다른 프로젝트 설정값일 수 있음)와
#    합쳐 엉뚱한 위치(runs/detect/htp_yolo/runs/...)에 저장해버린다. (실제 발생한 함정)
OUTPUT_ROOT = Path("htp_yolo").resolve()         # 산출물 루트 (htp_finetune.ipynb와 공유)
QD_RAW_DIR  = OUTPUT_ROOT / "quickdraw_raw"      # Quick Draw 원본 ndjson 캐시
QD_DS_DIR   = OUTPUT_ROOT / "stageA_quickdraw"   # 합성 데이터셋
RUNS_DIR    = OUTPUT_ROOT / "runs"
RUN_NAME    = "stageA_quickdraw_v4"              # 학습 로그 폴더명 (v1·v2·v3와 분리)

# ---- 산출물 이름 ----
#   클래스를 확장해 새 모델을 만들 때는 반드시 이 이름을 바꾼다.
#   기존 sketch_base.pt는 현재 htp_best.pt의 시작 가중치라 덮어쓰면 계보가 끊긴다.
#   v1 sketch_base.pt(33종) · v2 v2.pt(100종) · v3 v3.pt(101종)는 그대로 보존된다.
SKETCH_MODEL_NAME = "sketch_base_v4.pt"   # v3에서 school bus를 bus로 병합 → 100클래스
SKETCH_BASE       = OUTPUT_ROOT / SKETCH_MODEL_NAME

FORCE_RETRAIN = False   # True면 SKETCH_BASE가 이미 있어도 다시 학습해 덮어쓴다

# ---- 학습 하이퍼파라미터 ----
MODEL_ARCH = "yolo11s.pt"   # 스모크: yolo11n / 본학습: yolo11s (또는 yolo11m)
IMG_SIZE   = 960            # 스모크: 640 / 본학습: 960~1280
BATCH      = 16
WORKERS    = 8
DEVICE     = 0 if CUDA_AVAILABLE else "cpu"   # 오버라이드: "cpu" / 0 / "0,1"
SEED       = 42

# ---- 추가 튜닝: Ultralytics train() 인자를 자유롭게 전달 ----
# 예) {"cos_lr": True, "lr0": 0.001, "optimizer": "AdamW", "mosaic": 0.5, "close_mosaic": 10}
# freeze는 넣지 않는다 — Stage A는 COCO에서 처음부터 학습하는 단계라 백본을 얼릴 이유가 없다
# (freeze는 htp_finetune.ipynb의 파인튜닝 전용).
STAGE_A_TRAIN_KW = {}

# ---- 합성 데이터셋 ----
QD_EPOCHS          = 40      # v2·v3가 40으로 학습됐다 — 세대 간 비교하려면 바꾸지 말 것
QD_PER_CATEGORY    = 15000   # 카테고리당 내려받을 그림 수 (스모크는 800으로 낮출 것)
QD_TRAIN_IMAGES    = 40000   # 합성 학습 이미지 수 (클래스 100종 → 클래스당 ~2,400 인스턴스). 스모크는 1500
QD_VAL_IMAGES      = 200     # ⚠️ train 대비 0.5%다. 클래스당 val 인스턴스가 ~12개뿐이라
                             #    per-class mAP가 매우 흔들린다. 클래스별 비교로 판단할
                             #    거라면 2000 정도로 올릴 것(v2·v3는 200으로 측정됨).
QD_CANVAS          = 640     # 합성 캔버스 해상도
QD_OBJ_PER_IMAGE   = (3, 9)  # 이미지당 객체 수 범위
QD_ONLY_RECOGNIZED = True    # Quick Draw가 '인식 성공'으로 표시한 그림만 사용(품질 필터)
QD_BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/simplified/"

# Quick Draw 카테고리 100종 — 만 12세 이하 아동 자유화 기준으로 선정, URL 존재 전수 검증 완료.
#   앞 33종: HTP 요소 대응(sketch_base.pt와 동일). 그 뒤 67종: 동물·탈것·음식·자연·사물 확장.
#   HTP에 없는 요소(window/roof/chimney/smoke/hair/sneaker)는 Quick Draw에도 없어 제외.
# ⚠️ 카테고리명은 Quick Draw 공식 표기와 정확히 일치해야 한다(틀리면 다운로드 404).
# ⚠️ '사람 전체'에 해당하는 카테고리는 Quick Draw에 존재하지 않는다. face·arm·leg 같은
#    부위만 있다. person 클래스는 아래 HTP 스프라이트로 따로 만든다.
QD_CATEGORIES = [
    "house", "tree", "door", "fence", "flower", "grass",
    "bush", "sun", "cloud", "moon", "star", "bird",
    "squirrel", "mountain", "face", "eye", "nose", "mouth",
    "ear", "arm", "hand", "leg", "foot", "shoe",
    "sock", "t-shirt", "pants", "swing set", "ladder", "stairs",
    "river", "pool", "garden", "cat", "dog", "rabbit",
    "bear", "lion", "tiger", "elephant", "giraffe", "monkey",
    "pig", "cow", "horse", "sheep", "duck", "penguin",
    "owl", "fish", "butterfly", "bee", "spider", "frog",
    "snake", "car", "bus", "truck", "train", "airplane",
    "helicopter", "bicycle", "school bus", "firetruck", "apple", "banana",
    "strawberry", "watermelon", "carrot", "ice cream", "cake", "pizza",
    "hamburger", "cookie", "donut", "lollipop", "rainbow", "snowman",
    "snowflake", "lightning", "palm tree", "leaf", "chair", "table",
    "bed", "clock", "book", "cup", "umbrella", "pencil",
    "hat", "castle", "bridge", "smiley face", "crown", "soccer ball",
    "basketball", "skateboard", "guitar", "piano",
]

# ---- 카테고리 → 클래스 병합 ----
# 손그림에서 사실상 구분이 안 되는 카테고리를 한 클래스로 합친다.
# 카테고리는 그대로 내려받아 쓰고(그림 다양성 유지) **라벨만** 합친 이름으로 단다.
#
# 왜: v3에서 bus 정확도가 눈에 띄게 떨어졌다. 손그림의 bus와 school bus는
#     '네모 + 바퀴 둘'로 거의 같아 서로를 갉아먹는다. 아동 그림 분석에 둘의 구분이
#     필요하지도 않다. 혼동 쌍을 없애는 이득이 클래스 하나 잃는 손해보다 크다.
# ⚠️ 합치면 한 클래스에 카테고리가 2개 붙는다. 카테고리를 균등하게 뽑으면 bus만
#     2배 과대표집되므로, make_canvas는 '클래스 균등 → 그 안에서 카테고리 선택'
#     순서로 뽑는다(아래 QD_CLS_CATS).
# 더 볼 만한 후보: firetruck↔truck(같은 문제). palm tree↔tree는 생김새가 뚜렷이
#     달라 합치지 않았다.
QD_CLASS_MAP = {
    "school bus": "bus",
}

# ═══════════ HTP 사람 스프라이트 → person 클래스 (S15P11B209-685) ═══════════
# Quick Draw에 '사람 전체' 카테고리가 없어 v2까지는 person을 만들 수 없었다.
# HTP 데이터셋의 `사람전체` bbox를 크롭해 Quick Draw 스케치와 똑같은 방식으로
# 합성 캔버스에 붙인다. 남자사람·여자사람을 합쳐 단일 person 클래스로 쓴다
# (성별 구분은 범용 탐지 목적에 불필요하고, 합치면 표본이 22,400장으로 2배가 된다).
#
# ⚠️ 이 노트북의 'HTP 없이 단독 실행' 성질은 유지된다 — HTP를 못 찾으면 경고만 내고
#    person 없이 100클래스로 끝까지 돌아간다.
HTP_SPRITE_ENABLE = True           # False면 HTP가 있어도 쓰지 않는다(v2와 동일한 100클래스)
HTP_PERSON_CLASS  = "person"
# ⚠️ 스프라이트 1개가 부위 라벨을 평균 17개(눈2·팔2·손2·다리2·발2·귀2·얼굴·코·입·상체)
#    끌고 온다. 이 값을 올리면 person만 느는 게 아니라 그 부위 클래스가 함께 폭증해
#    다른 클래스의 학습 신호를 빼앗는다. v3(0.15 · 40,000장)에서 eye·arm 등이 다른
#    클래스의 5배가 되어 bus 같은 약한 클래스가 무너졌다.
#    균형점 ≈ (클래스당 기대 인스턴스) / QD_TRAIN_IMAGES. 40,000장이면 0.06 정도가
#    person ≈ 2,400개로 다른 클래스와 같은 수준이 된다. 아래 실행 결과의 경고를 볼 것.
HTP_PERSON_PROB   = 0.06           # 40,000장 → person ≈ 2,400개 (다른 클래스와 같은 수준)
HTP_PERSON_SCALE  = (0.25, 0.70)   # 캔버스 대비 스프라이트 '긴 변' 비율
HTP_POOL_TRAIN    = 1200           # 메모리에 올려둘 스프라이트 수 (train 캔버스용)
HTP_POOL_VAL      = 200            # val 캔버스용 — HTP Validation 분할에서만 뽑아 누수를 막는다
HTP_INK_FLOOR     = 25             # 이 미만 잉크 강도는 종이 잡티로 보고 지운다 (0~255)
HTP_SPRITE_MAXPX  = 512            # 풀에 저장할 스프라이트 긴 변 상한 (메모리 절약)
HTP_SUB_LABELS    = True           # 사람 안의 눈·팔·손…을 함께 라벨한다 (아래 설명 필독)

# HTP 라벨 → Stage A 클래스.
# ⚠️ `사람전체` bbox 안에는 하위 객체가 평균 22개 들어 있다(눈 2·팔 2·손 2·다리 2·발 2·
#    얼굴·코·입·귀 2·상체 …). person 박스만 달고 붙이면 그것들이 전부 '라벨 없는 배경'이
#    되어, 이미 갖고 있는 eye/arm/hand/leg/foot/face 클래스에 false negative를 대량 주입한다.
#    그래서 하위 라벨을 함께 옮긴다(HTP_SUB_LABELS=True). 크롭→스케일→붙이기가 동일한
#    어파인 변환이라 하위 bbox도 그대로 따라온다. 부위 클래스가 Quick Draw 낙서뿐 아니라
#    실제 아동 그림으로도 학습되는 이득이 덤으로 붙는다.
#    대응 클래스가 없는 머리·머리카락·목·단추·주머니는 버린다 — 오염될 클래스가 없다
#    (머리 영역은 얼굴이 대부분 덮는다).
HTP_LABEL_MAP = {
    "얼굴": "face", "눈": "eye", "코": "nose", "입": "mouth", "귀": "ear",
    "팔": "arm", "손": "hand", "다리": "leg", "발": "foot", "상체": "t-shirt",
    "남자구두": "shoe", "여자구두": "shoe", "운동화": "shoe",
}

# 경로 탐색 규칙은 htp_finetune.ipynb와 동일하게 둔다 — 폴더/zip 이름이 바뀌어도 견디게 한다.
def _find_data_root() -> Path:
    here = Path.cwd()
    for base in [here, *here.parents]:
        for cand in (base / "data", base / "ai" / "models" / "data"):
            if cand.exists():
                return cand
    return here / "data"

def _find_htp_roots(data_root: Path):
    # Training/ 과 Validation/ 을 모두 가진 폴더를 하위 2단계까지 '전부' 모은다.
    # ⚠️ 후보는 하나가 아니다 — AI-Hub 원본(zip)과 압축 해제본이 함께 있는 환경이 실제로 있다.
    #    (data/HTP/01-1.정식개방데이터/ 와 data/prepared/htp/ 가 둘 다 Training·Validation을 갖는다)
    #    어느 쪽이 실제로 읽히는지는 스프라이트 셀이 사람 그림 개수로 검증해 고른다.
    roots, seen = [], set()
    cands = [data_root]
    if data_root.exists():
        for p in data_root.iterdir():
            if p.is_dir():
                cands.append(p)
                cands.extend(q for q in p.iterdir() if q.is_dir())
    for c in cands:
        if c not in seen and (c / "Training").is_dir() and (c / "Validation").is_dir():
            seen.add(c); roots.append(c)
    return roots

# 사람 그림 폴더. 한글 이름 '정확 일치'로 찾는다.
#   ⚠️ 부분문자열로 판정하면 안 된다 — zip 표기(Man.zip / Woman.zip)에서 woman ⊃ man 이라
#      여자사람이 남자사람으로 잘못 분류된다(htp_finetune.ipynb `_zip_type`의 함정).
HTP_PERSON_DIRS = ("남자사람", "여자사람")

HTP_ROOTS = _find_htp_roots(_find_data_root()) if HTP_SPRITE_ENABLE else []

random.seed(SEED); np.random.seed(SEED)
for d in (OUTPUT_ROOT, QD_RAW_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 클래스 수 대비 합성 이미지가 너무 적으면 희귀 클래스가 학습되지 않는다.
# 이미지당 평균 객체 수 기준으로 '클래스당 기대 인스턴스'를 추정해 경고한다.
_qd_classes = {QD_CLASS_MAP.get(c, c) for c in QD_CATEGORIES}
_n_classes  = len(_qd_classes) + (1 if HTP_ROOTS else 0)
_avg_obj = sum(QD_OBJ_PER_IMAGE) / 2
_per_cls = QD_TRAIN_IMAGES * _avg_obj / max(_n_classes, 1)

print("OUTPUT_ROOT :", OUTPUT_ROOT)
print("산출물      :", SKETCH_BASE, "(이미 있음)" if SKETCH_BASE.exists() else "(신규)")
print(f"arch={MODEL_ARCH} imgsz={IMG_SIZE} batch={BATCH} device={DEVICE} epochs={QD_EPOCHS}")
print(f"카테고리 {len(QD_CATEGORIES)}개 · 합성 train {QD_TRAIN_IMAGES}장 / val {QD_VAL_IMAGES}장")
if QD_CLASS_MAP:
    print("카테고리 병합:", ", ".join(f"{k} → {v}" for k, v in QD_CLASS_MAP.items()),
          f"(클래스 {len(QD_CATEGORIES)} → {len(_qd_classes)})")
if HTP_ROOTS:
    print(f"HTP 후보 {len(HTP_ROOTS)}곳 — 스프라이트 셀에서 실제로 읽히는 곳을 고른다:")
    for _r in HTP_ROOTS:
        print("   ", _r)
    print(f"→ 성공하면 '{HTP_PERSON_CLASS}' 클래스 추가 (총 {_n_classes}종)")
elif HTP_SPRITE_ENABLE:
    print("⚠️ HTP 데이터셋을 찾지 못했습니다 — person 없이 "
          f"{len(_qd_classes)}클래스로 진행합니다(Quick Draw만).")
else:
    print(f"HTP 스프라이트 비활성(HTP_SPRITE_ENABLE=False) — {len(_qd_classes)}클래스")
print(f"클래스당 기대 인스턴스 ≈ {_per_cls:.0f}개", end="  ")
print("⚠️ 200 미만 — QD_TRAIN_IMAGES를 올리세요" if _per_cls < 200 else "✓")
_htp_est = int(QD_TRAIN_IMAGES * HTP_PERSON_PROB)
if HTP_ROOTS:
    print(f"person 예상 {_htp_est}개 = 클래스당 기대치의 {_htp_est / max(_per_cls, 1):.1f}배", end="  ")
    print("⚠️ 부위 클래스(eye·arm…)가 이 배수의 2배로 함께 늘어 다른 클래스를 압도한다."
          " HTP_PERSON_PROB를 낮추세요" if _htp_est > 1.5 * _per_cls else "✓")
if SKETCH_BASE.exists() and not FORCE_RETRAIN:
    print("→ 학습 셀은 기존 가중치를 재사용한다(다시 학습하려면 FORCE_RETRAIN=True).")

### Quick Draw ↔ HTP 클래스 대응 (참고)

Stage A와 Stage B는 **클래스 수가 다르므로 검출 헤드는 재초기화**된다.
전이되는 가치는 **백본/넥이 학습한 선화(line-art) 특징**이며, 이는 의도된 설계다.
아래 표는 "Stage A가 HTP에 얼마나 관련된 것을 배우는가"를 보여주는 참고자료일 뿐,
학습에는 쓰이지 않는다.

In [ ]:
# ── Quick Draw → HTP 대응 참고표 ──────────────────────────────────
QD_TO_HTP = {
    "house": "집전체", "tree": "나무전체", "door": "문", "fence": "울타리",
    "flower": "꽃", "grass": "잔디", "bush": "잔디(유사)", "sun": "태양",
    "cloud": "구름", "moon": "달", "star": "별", "bird": "새",
    "squirrel": "다람쥐", "mountain": "산", "face": "얼굴", "eye": "눈",
    "nose": "코", "mouth": "입", "ear": "귀", "arm": "팔", "hand": "손",
    "leg": "다리", "foot": "발", "shoe": "남자구두/여자구두", "sock": "발(유사)",
    "t-shirt": "상체", "pants": "다리(유사)", "swing set": "그네",
    "pool": "연못", "river": "길(유사)", "ladder": "(보조)",
    "stairs": "(보조)", "garden": "(보조)",
}
display(pd.DataFrame([{"QuickDraw": c, "HTP 대응": QD_TO_HTP.get(c, "(보조)")}
                      for c in QD_CATEGORIES]))

# HTP 클래스 목록은 htp_finetune.ipynb가 만든다. 있으면 커버리지를 계산하고,
# 없으면 조용히 건너뛴다 — 이 노트북은 HTP 산출물에 의존하지 않는다.
_cls_json = OUTPUT_ROOT / "classes.json"
if _cls_json.exists():
    _htp_names = json.loads(_cls_json.read_text(encoding="utf-8"))["names"]
    _covered = {v for v in QD_TO_HTP.values() if v in _htp_names}
    print(f"HTP {len(_htp_names)}개 중 Quick Draw로 직접 대응되는 클래스: {len(_covered)}개")
    print("대응 없는 HTP 클래스(예):",
          ", ".join(sorted(set(_htp_names) - _covered)[:15]), "...")
else:
    print("classes.json 없음 — 커버리지 계산 생략 (htp_finetune.ipynb 실행 시 생성됨)")

## 3. 합성 데이터셋 생성  🎨

### 핵심 아이디어
Quick Draw 원본은 **한 그림에 객체 1개**이고 좌표가 0–255를 꽉 채우도록 정규화되어 있어,
그대로 쓰면 bbox가 항상 이미지 전체가 된다(퇴화 과제). 따라서:

1. 카테고리별 스케치를 스트리밍으로 내려받는다(필요한 만큼만 읽고 조기 종료 → 전체 다운로드 불필요).
2. 스케치를 **랜덤 크기로 렌더링**해 빈 캔버스의 **랜덤 위치에 합성**한다.
3. 배치를 우리가 하므로 **bbox를 정확히 계산**할 수 있다(잉크 픽셀 min/max).
4. 연필 느낌에 가깝도록 선 두께·농도·약간의 회전을 무작위화해 도메인 격차를 줄인다.

In [ ]:
# ── Quick Draw 다운로드 (필요한 줄만 스트리밍, 캐시) ──────────────
def qd_download(category: str, n: int, dest_dir: Path) -> Path:
    dest = dest_dir / f"{category.replace(' ', '_')}.ndjson"
    if dest.exists():
        with dest.open(encoding="utf-8") as f:
            if sum(1 for _ in f) >= n:
                return dest   # 캐시 충분
    url = QD_BASE_URL + quote(category) + ".ndjson"
    lines = []
    req = urllib.request.Request(url, headers={"User-Agent": "htp-baseline/1.0"})
    with urllib.request.urlopen(req, timeout=180) as r:
        for raw in r:                      # ndjson: 줄 단위 스트리밍
            lines.append(raw.decode("utf-8"))
            if len(lines) >= n:
                break                      # 필요한 만큼만 받고 조기 종료
    dest.write_text("".join(lines), encoding="utf-8")
    return dest

def qd_load(category: str, n: int) -> list:
    # 스트로크 리스트 반환. 각 스트로크는 [[x...], [y...]] (0~255 좌표계)
    p = QD_RAW_DIR / f"{category.replace(' ', '_')}.ndjson"
    out = []
    with p.open(encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            if QD_ONLY_RECOGNIZED and not d.get("recognized", True):
                continue
            out.append(d["drawing"])
            if len(out) >= n:
                break
    return out

QD_POOL = {}
for cat in tqdm(QD_CATEGORIES, desc="Quick Draw 다운로드"):
    qd_download(cat, QD_PER_CATEGORY, QD_RAW_DIR)
    QD_POOL[cat] = qd_load(cat, QD_PER_CATEGORY)

# QD_NAMES는 'Quick Draw 스케치를 뽑을 수 있는 카테고리' 목록이다.
# ⚠️ 최종 클래스 목록이 아니다 — 병합(QD_CLASS_MAP)으로 줄고 HTP person이 뒤 셀에서
#    추가된다. 학습에 쓰는 id는 CLASS_NAMES / CLASS_ID 다.
#    여기서 id를 만들면 병합·person 반영 후 어긋난다.
QD_NAMES = sorted(QD_POOL)                         # 결정론적 정렬

# 카테고리 → 클래스, 클래스 → 카테고리들. 병합이 없으면 1:1이라 기존과 동일하게 동작한다.
QD_CLS_OF   = {c: QD_CLASS_MAP.get(c, c) for c in QD_NAMES}
QD_CLS_CATS = defaultdict(list)
for _c in QD_NAMES:
    QD_CLS_CATS[QD_CLS_OF[_c]].append(_c)
QD_CLASSES = sorted(QD_CLS_CATS)                   # Quick Draw 쪽 클래스 목록

print(f"\n카테고리 {len(QD_NAMES)}개, 그림 총 {sum(len(v) for v in QD_POOL.values()):,}개 확보")
print("카테고리별 개수 예시:", {c: len(QD_POOL[c]) for c in QD_NAMES[:5]})
_merged = {k: v for k, v in QD_CLS_CATS.items() if len(v) > 1}
print(f"Quick Draw 클래스 {len(QD_CLASSES)}개"
      + (f" (병합됨: {_merged})" if _merged else " (병합 없음)"))

In [ ]:
# ── HTP 사람 스프라이트 풀 (person 클래스) ────────────────────────
# Quick Draw에 없는 person을 HTP `사람전체` bbox 크롭으로 만든다.
# 산출 형식은 render_sketch와 맞춘다 — (L 모드 패치, 패치 좌상단 기준 박스들).
#
# 레이아웃 두 가지를 모두 지원한다(htp_finetune.ipynb와 같은 규칙):
#   (A) 폴더: <root>/<split>/<유형>/**.jpg,*.json     ← 압축 해제본
#   (B) zip : <root>/<split>/(원천|라벨)/*.zip        ← AI-Hub 원본
# 파일 참조(ref)는 폴더면 Path, zip이면 (zip경로, 멤버명)으로 통일하고 _read_ref가 흡수한다.
import zipfile

def _fix_name(name: str) -> str:
    # zip 내부 이름이 cp437로 깨진 경우만 복원. 정상 이름은 그대로 둔다.
    if "�" in name:
        try:
            return name.encode("cp437").decode("utf-8")
        except Exception:
            pass
    return name

def _person_type(stem: str):
    # 파일/zip 이름에서 사람 유형 판정. 사람이 아니면 None.
    # ⚠️ 반드시 여자사람(woman)을 먼저 검사한다 — woman ⊃ man.
    s = stem.lower()
    if "여자사람" in s or "woman" in s:
        return "여자사람"
    if "남자사람" in s or "man" in s:
        return "남자사람"
    return None

def _htp_person_files(root: Path, split: str):
    """(이미지 ref, 라벨 ref) 쌍 목록. 사람 그림만 고른다."""
    sdir = root / split
    if not sdir.is_dir():
        return []

    # (A) 폴더 모드 — <split>/<유형>/ 아래에 images·labels 하위폴더가 있어도,
    #     파일이 바로 놓여 있어도 동작하도록 rglob으로 훑는다.
    out = []
    for folder in HTP_PERSON_DIRS:
        tdir = sdir / folder
        if not tdir.is_dir():
            continue
        lbls = {p.stem: p for p in tdir.rglob("*.json")}
        for ip in sorted(tdir.rglob("*.jpg")):
            lp = lbls.get(ip.stem)
            if lp is not None:
                out.append((ip, lp))
    if out:
        return out

    # (B) zip 모드 — 원천/라벨 zip에서 사람 유형 zip만 열어 멤버를 인덱싱한다.
    def _subdirs(keys):
        return [p for p in sorted(sdir.iterdir())
                if p.is_dir() and any(k in p.name.lower() for k in keys)]

    def _index(keys, ext, reg):
        for d in _subdirs(keys):
            for zp in sorted(d.glob("*.zip")):
                if _person_type(zp.stem) is None:
                    continue                      # 집·나무 zip은 열지 않는다
                with zipfile.ZipFile(zp) as z:
                    for m in z.namelist():
                        if m.lower().endswith(ext):
                            reg[Path(_fix_name(m)).stem] = (zp, m)

    imgs, lbls = {}, {}
    _index(("원천", "source", "image", "img"), ".jpg", imgs)
    _index(("라벨", "label"),                  ".json", lbls)
    for stem in sorted(imgs):
        if stem in lbls and _person_type(stem) is not None:
            out.append((imgs[stem], lbls[stem]))
    return out

_ZIP_CACHE = {}
def _read_ref(ref) -> bytes:
    if isinstance(ref, tuple):
        zp, member = ref
        z = _ZIP_CACHE.get(zp)
        if z is None:
            z = _ZIP_CACHE[zp] = zipfile.ZipFile(zp)   # 파일마다 여는 건 너무 느리다
        return z.read(member)
    return Path(ref).read_bytes()


def extract_person_sprite(img_ref, lbl_ref):
    """HTP 그림 1장 → (스프라이트, [(클래스명, x1, y1, x2, y2), ...]). 실패하면 None."""
    try:
        ann = json.loads(_read_ref(lbl_ref).decode("utf-8"))["annotations"]["bbox"]
    except Exception:
        return None
    person = next((b for b in ann if b.get("label") == "사람전체"), None)
    if person is None:
        return None

    try:
        im = Image.open(io.BytesIO(_read_ref(img_ref))).convert("L")
    except Exception:
        return None
    W, H = im.size
    px1 = max(int(person["x"]), 0)
    py1 = max(int(person["y"]), 0)
    px2 = min(int(person["x"]) + int(person["w"]), W)
    py2 = min(int(person["y"]) + int(person["h"]), H)
    if px2 - px1 < 48 or py2 - py1 < 48:
        return None
    crop = im.crop((px1, py1, px2, py2))

    # 잉크 추출 — 종이 밝기를 기준으로 어두운 쪽만 남기고 대비를 편다.
    # HTP는 연필 그림이라 명도만으로 충분하다(실측: 종이 강도 중앙값 2/255, 선 최대 195~234).
    # 색연필까지 살려야 하면 color_ablation_test.ipynb의 ink_normalize(채도 병용)를 참고.
    arr = np.asarray(crop, dtype=np.float32)
    paper = float(np.percentile(arr, 95))            # 종이 밝기
    strength = np.clip(paper - arr, 0.0, None)       # 0=종이, 클수록 진한 선
    peak = float(strength.max())
    if peak < 1.0:
        return None                                   # 빈 크롭
    strength *= 255.0 / peak
    strength[strength < HTP_INK_FLOOR] = 0.0          # 종이 잡티·JPEG 노이즈 제거
    sprite = Image.fromarray((255.0 - strength).astype(np.uint8), mode="L")

    # person 박스는 크롭 전체. 하위 라벨은 '중심이 person bbox 안'인 것만 옮긴다
    # — 따로 떨어뜨려 그린 신발(운동화·구두)은 이 조건에서 자연히 빠진다.
    boxes = [(HTP_PERSON_CLASS, 0.0, 0.0, float(px2 - px1), float(py2 - py1))]
    if HTP_SUB_LABELS:
        for b in ann:
            name = HTP_LABEL_MAP.get(b.get("label"))
            if name is None:
                continue
            bx1, by1 = float(b["x"]), float(b["y"])
            bx2, by2 = bx1 + float(b["w"]), by1 + float(b["h"])
            cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
            if not (px1 <= cx <= px2 and py1 <= cy <= py2):
                continue
            qx1, qy1 = max(bx1, px1) - px1, max(by1, py1) - py1
            qx2, qy2 = min(bx2, px2) - px1, min(by2, py2) - py1
            if qx2 - qx1 >= 2 and qy2 - qy1 >= 2:
                boxes.append((name, qx1, qy1, qx2, qy2))

    # 어차피 캔버스에 축소해 붙으므로 긴 변을 상한에 맞춰 미리 줄여 메모리를 아낀다.
    w, h = sprite.size
    if max(w, h) > HTP_SPRITE_MAXPX:
        k = HTP_SPRITE_MAXPX / max(w, h)
        sprite = sprite.resize((max(1, round(w * k)), max(1, round(h * k))), Image.LANCZOS)
        boxes = [(n, x1 * k, y1 * k, x2 * k, y2 * k) for n, x1, y1, x2, y2 in boxes]
    return sprite, boxes


def build_person_pool(files, n: int, seed: int, desc: str):
    if not files:
        return []
    files = list(files)
    random.Random(seed).shuffle(files)     # 남자·여자·연령이 고루 섞이도록
    pool = []
    for img_ref, lbl_ref in tqdm(files, desc=desc, leave=False):
        s = extract_person_sprite(img_ref, lbl_ref)
        if s is not None:
            pool.append(s)
        if len(pool) >= n:
            break
    return pool


# ── HTP 루트 선택 ────────────────────────────────────────────────
# 후보를 순서대로 열어보고 '사람 그림이 실제로 읽히는' 첫 곳을 쓴다.
# Training/Validation 폴더만 보고 고르면 AI-Hub 원본과 압축 해제본이 함께 있을 때
# 엉뚱한 쪽이 잡힌다(실제로 발생: 원본 zip 루트가 먼저 선택돼 스프라이트 0개).
HTP_ROOT, HTP_FILES = None, {}
for _cand in HTP_ROOTS:
    _tr = _htp_person_files(_cand, "Training")
    if _tr:
        HTP_ROOT = _cand
        HTP_FILES = {"Training": _tr, "Validation": _htp_person_files(_cand, "Validation")}
        print(f"HTP 루트 선택: {_cand}")
        print(f"  사람 그림 Training {len(_tr):,}장 / "
              f"Validation {len(HTP_FILES['Validation']):,}장")
        break
    print(f"  (건너뜀) 사람 그림이 없는 HTP 후보: {_cand}")
HTP_AVAILABLE = HTP_ROOT is not None

# train 캔버스는 HTP Training에서, val 캔버스는 HTP Validation에서만 뽑는다.
# 같은 사람 그림이 양쪽에 들어가면 val 지표가 부풀려진다(누수).
HTP_POOL = {"train": [], "val": []}
if HTP_AVAILABLE:
    HTP_POOL["train"] = build_person_pool(HTP_FILES["Training"],   HTP_POOL_TRAIN,
                                          SEED,     "HTP person 스프라이트(train)")
    HTP_POOL["val"]   = build_person_pool(HTP_FILES["Validation"], HTP_POOL_VAL,
                                          SEED + 1, "HTP person 스프라이트(val)")
    if not HTP_POOL["train"]:
        print("⚠️ 사람 그림은 찾았지만 `사람전체` 스프라이트를 하나도 만들지 못했습니다"
              " — person 없이 진행합니다.")
elif HTP_ROOTS:
    print("⚠️ HTP 후보를 모두 열어봤지만 사람 그림을 찾지 못했습니다 — person 없이 진행합니다.")

# ── 최종 클래스 목록 ──────────────────────────────────────────────
# 카테고리(QD_NAMES)가 아니라 '병합 후 클래스'(QD_CLASSES)에 person을 더한 것이다.
# QD_POOL에는 person이 없으므로 스케치를 고를 때는 계속 QD_CLS_CATS/QD_POOL을 쓴다.
EXTRA_NAMES = [HTP_PERSON_CLASS] if HTP_POOL["train"] else []
CLASS_NAMES = sorted(QD_CLASSES + EXTRA_NAMES)   # 결정론적 정렬
CLASS_ID    = {c: i for i, c in enumerate(CLASS_NAMES)}

print(f"HTP 스프라이트: train {len(HTP_POOL['train'])}개 / val {len(HTP_POOL['val'])}개")
if HTP_POOL["train"]:
    _sub = Counter(n for _, bxs in HTP_POOL["train"] for n, *_ in bxs)
    print("  스프라이트가 실어오는 라벨(train 풀 합계):",
          ", ".join(f"{k}={v}" for k, v in _sub.most_common()))
    print(f"  → 캔버스당 {HTP_PERSON_PROB:.0%} 확률로 주입 시 person ≈ "
          f"{int(QD_TRAIN_IMAGES * HTP_PERSON_PROB)}개 예상")
print(f"최종 클래스 {len(CLASS_NAMES)}개"
      + (f" (person 포함, id={CLASS_ID[HTP_PERSON_CLASS]})" if EXTRA_NAMES else " (person 없음)"))

In [ ]:
# ── 스케치 렌더링 + 합성 캔버스 생성 ──────────────────────────────
def render_sketch(strokes, size: int, rng: random.Random):
    # 스케치를 size x size 패치(L 모드, 흰 배경)에 렌더링.
    # 반환: (패치, 패치 내 tight bbox (x1,y1,x2,y2))
    pts = [(x, y) for s in strokes for x, y in zip(s[0], s[1])]
    if len(pts) < 2:
        return None, None
    xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
    x0, x1 = min(xs), max(xs); y0, y1 = min(ys), max(ys)
    sw = max(1, int(round(size / rng.choice([70, 90, 120]))))   # 선 두께 무작위
    pad = sw + 2
    span = max(x1 - x0, y1 - y0, 1)
    scale = (size - 2 * pad) / span

    patch = Image.new("L", (size, size), 255)
    dr = ImageDraw.Draw(patch)
    ink = rng.randint(40, 110)          # 연필 농도 무작위
    def tx(x, y):
        return (pad + (x - x0) * scale, pad + (y - y0) * scale)
    for s in strokes:
        pl = [tx(x, y) for x, y in zip(s[0], s[1])]
        if len(pl) == 1:
            dr.ellipse([pl[0][0]-sw, pl[0][1]-sw, pl[0][0]+sw, pl[0][1]+sw], fill=ink)
        else:
            dr.line(pl, fill=ink, width=sw, joint="curve")
    if rng.random() < 0.5:              # 약간의 회전
        patch = patch.rotate(rng.uniform(-8, 8), resample=Image.BILINEAR, fillcolor=255)

    # tight bbox는 렌더링이 모두 끝난 뒤 '실제 잉크 픽셀'에서 계산한다.
    # (좌표 계산으로 구하면 회전이 적용됐을 때 잉크가 박스 밖으로 삐져나간다)
    arr = np.asarray(patch)
    ink_px = np.argwhere(arr < 250)     # 배경 255, 안티에일리어싱 여유
    if ink_px.size == 0:
        return None, None
    (iy1, ix1), (iy2, ix2) = ink_px.min(0), ink_px.max(0)
    bb = (float(ix1), float(iy1), float(ix2 + 1), float(iy2 + 1))
    return patch, bb

def paste_person(canvas, boxes, pool, rng: random.Random):
    # HTP 사람 스프라이트 1개를 캔버스에 붙이고 라벨(person + 하위 부위)을 boxes에 추가한다.
    # ⚠️ 회전을 넣지 않는다 — 하위 bbox를 좌표 변환으로 함께 옮기므로 돌리면 박스가 어긋난다.
    #    (render_sketch는 잉크 픽셀에서 bbox를 다시 구하므로 회전이 안전하지만, 여기는 다르다)
    sprite, sboxes = rng.choice(pool)
    C = QD_CANVAS
    sw, sh = sprite.size
    k = (rng.uniform(*HTP_PERSON_SCALE) * C) / max(sw, sh)
    nw, nh = max(8, round(sw * k)), max(8, round(sh * k))
    if nw > C or nh > C:                                  # 캔버스를 넘지 않게 보정
        k = min(C / sw, C / sh)
        nw, nh = max(8, int(sw * k)), max(8, int(sh * k))
    patch = sprite.resize((nw, nh), Image.LANCZOS)

    # ⚠️ 축소하면 얇은 연필선이 픽셀 평균에 묻혀 흐려진다(1280px 원본 → 캔버스에서 ~200px).
    #    Quick Draw 스프라이트는 목표 크기에 맞춰 선을 '다시 그리므로' 이 손실이 없다.
    #    그래서 축소 뒤 잉크 강도를 다시 편다 — 안 하면 사람만 유령처럼 옅게 붙는다.
    parr = np.asarray(patch, dtype=np.float32)
    ink = 255.0 - parr                       # 0=종이, 클수록 진한 선
    peak = float(ink.max())
    if peak > 1.0:
        ink *= 255.0 / peak
    dens = rng.uniform(0.65, 1.0)            # 연필 농도 무작위화 (Quick Draw ink 40~110과 톤 맞춤)
    patch = Image.fromarray(np.clip(255.0 - ink * dens, 0, 255).astype(np.uint8), mode="L")

    px, py = rng.randint(0, C - nw), rng.randint(0, C - nh)
    region = canvas.crop((px, py, px + nw, py + nh))
    canvas.paste(ImageChops.darker(region, patch), (px, py))   # 겹침 시 진한 쪽 유지

    for name, x1, y1, x2, y2 in sboxes:
        cid = CLASS_ID.get(name)
        if cid is None:
            continue
        ax1, ay1 = x1 * k + px, y1 * k + py
        ax2, ay2 = x2 * k + px, y2 * k + py
        if ax2 - ax1 >= 4 and ay2 - ay1 >= 4:
            boxes.append((cid, ax1, ay1, ax2, ay2))

def make_canvas(rng: random.Random, person_pool=()):
    # Quick Draw 스케치를 랜덤 배치한 합성 장면 1장 생성.
    # person_pool이 주어지면 HTP 사람 스프라이트를 확률적으로 함께 붙인다.
    C = QD_CANVAS
    canvas = Image.new("L", (C, C), rng.randint(248, 255))   # 살짝 종이톤
    boxes = []
    # 사람을 먼저 붙인다 — 뒤이어 붙는 Quick Draw 객체가 위에 겹치며 가려짐이 생긴다.
    if person_pool and rng.random() < HTP_PERSON_PROB:
        paste_person(canvas, boxes, person_pool, rng)
    for _ in range(rng.randint(*QD_OBJ_PER_IMAGE)):
        # ⚠️ 카테고리가 아니라 '클래스'를 균등하게 뽑는다. 카테고리 균등으로 뽑으면
        #    병합된 클래스(bus ← bus·school bus)만 2배 과대표집된다.
        cls = rng.choice(QD_CLASSES)
        cat = rng.choice(QD_CLS_CATS[cls])
        if not QD_POOL[cat]:
            continue
        strokes = rng.choice(QD_POOL[cat])
        s = rng.randint(int(C * 0.12), int(C * 0.42))
        patch, bb = render_sketch(strokes, s, rng)
        if patch is None:
            continue
        px, py = rng.randint(0, C - s), rng.randint(0, C - s)
        region = canvas.crop((px, py, px + s, py + s))
        canvas.paste(ImageChops.darker(region, patch), (px, py))  # 겹침 시 진한 쪽 유지
        x1, y1, x2, y2 = bb[0] + px, bb[1] + py, bb[2] + px, bb[3] + py
        if x2 - x1 >= 4 and y2 - y1 >= 4:
            boxes.append((CLASS_ID[cls], x1, y1, x2, y2))
    return canvas.convert("RGB"), boxes

def write_yolo_split(out_dir: Path, n_images: int, seed: int, desc: str, person_pool=()):
    img_d, lbl_d = out_dir / "images", out_dir / "labels"
    img_d.mkdir(parents=True, exist_ok=True); lbl_d.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    C = QD_CANVAS
    for i in tqdm(range(n_images), desc=desc, leave=False):
        img, boxes = make_canvas(rng, person_pool)
        img.save(img_d / f"qd_{i:06d}.jpg", quality=92)
        lines = [f"{c} {((x1+x2)/2)/C:.6f} {((y1+y2)/2)/C:.6f} {(x2-x1)/C:.6f} {(y2-y1)/C:.6f}"
                 for c, x1, y1, x2, y2 in boxes]
        (lbl_d / f"qd_{i:06d}.txt").write_text("\n".join(lines), encoding="utf-8")

qd_yaml = QD_DS_DIR / "data.yaml"

def _yaml_class_names(p: Path):
    # data.yaml의 "  0: arm" 줄에서 클래스 목록만 뽑는다. 없으면 None.
    if not p.exists():
        return None
    names = []
    for line in p.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if s[:1].isdigit() and ":" in s:
            names.append(s.split(":", 1)[1].strip())
    return names or None

# 캐시 판정: 이미지 수뿐 아니라 '클래스 목록이 같은지'까지 본다.
# (이미지 수만 보면 QD_CATEGORIES를 바꿔도 캐시가 그대로 쓰이는 함정이 있었다)
_train_img_dir = QD_DS_DIR / "train" / "images"
_enough = _train_img_dir.exists() and len(list(_train_img_dir.glob("*.jpg"))) >= QD_TRAIN_IMAGES
_same_classes = _yaml_class_names(qd_yaml) == CLASS_NAMES
if _enough and _same_classes:
    print("합성 데이터셋 캐시 사용:", QD_DS_DIR.resolve())
    print("  ⚠️ 클래스 외 합성 파라미터(QD_CANVAS 등)를 바꿨다면 이 폴더를 직접 지우세요.")
else:
    if _enough and not _same_classes:
        print("클래스 목록이 바뀌어 합성 데이터셋을 다시 만듭니다.")
    shutil.rmtree(QD_DS_DIR, ignore_errors=True)
    write_yolo_split(QD_DS_DIR / "train", QD_TRAIN_IMAGES, SEED,     "합성 train",
                     HTP_POOL["train"])
    write_yolo_split(QD_DS_DIR / "val",   QD_VAL_IMAGES,   SEED + 1, "합성 val",
                     HTP_POOL["val"])
    print("생성 완료:", QD_DS_DIR.resolve())

qd_yaml.parent.mkdir(parents=True, exist_ok=True)
qd_yaml.write_text(
    "path: " + str(QD_DS_DIR.resolve()) + "\ntrain: train/images\nval: val/images\n"
    + "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES)),
    encoding="utf-8")
print("data.yaml:", qd_yaml.resolve(), "| 클래스", len(CLASS_NAMES), "개")

# ── 클래스 분포 확인 ──────────────────────────────────────────────
# HTP 스프라이트가 부위 클래스(eye·arm·hand…)를 얼마나 밀어올렸는지 본다.
# 특정 클래스가 과대표집되면 HTP_PERSON_PROB를 낮추거나 HTP_SUB_LABELS를 끈다.
def label_counts(split_dir: Path):
    cnt = Counter()
    for p in (split_dir / "labels").glob("*.txt"):
        for line in p.read_text(encoding="utf-8").splitlines():
            if line.strip():
                cnt[CLASS_NAMES[int(line.split()[0])]] += 1
    return cnt

_cnt = label_counts(QD_DS_DIR / "train")
_dist = pd.DataFrame(sorted(_cnt.items(), key=lambda kv: -kv[1]),
                     columns=["클래스", "인스턴스"])
print(f"\ntrain 인스턴스 총 {sum(_cnt.values()):,}개 · 클래스 {len(_cnt)}개 "
      f"(중앙 {int(_dist['인스턴스'].median())}개)")
print("상위 10 / 하위 10:")
display(pd.concat([_dist.head(10), _dist.tail(10)]))
if HTP_PERSON_CLASS in _cnt:
    print(f"person: {_cnt[HTP_PERSON_CLASS]}개")

In [ ]:
# ── 합성 결과 검증: bbox를 이미지 위에 그려 확인 ──────────────────
# person이 들어간 장면을 우선 보여준다 — 하위 라벨(눈·팔·손…)이 스케일·오프셋 변환을
# 제대로 따라왔는지, 박스가 밀리지 않았는지 눈으로 확인하는 것이 이 셀의 핵심이다.
_all = sorted((QD_DS_DIR / "train" / "images").glob("*.jpg"))
_lbl_of = lambda ip: ip.parent.parent / "labels" / (ip.stem + ".txt")

smp = []
if HTP_PERSON_CLASS in CLASS_ID:
    _pid = str(CLASS_ID[HTP_PERSON_CLASS])
    for ip in _all[:400]:                       # 앞부분만 훑어도 확률상 충분히 나온다
        if any(ln.split()[:1] == [_pid] for ln in _lbl_of(ip).read_text().splitlines() if ln.strip()):
            smp.append(ip)
        if len(smp) >= 2:
            break
smp += [p for p in _all[:3] if p not in smp]
smp = smp[:3]

fig, axes = plt.subplots(1, len(smp), figsize=(6 * len(smp), 6))
if len(smp) == 1: axes = [axes]
for ax, ip in zip(axes, smp):
    im = Image.open(ip); W, H = im.size
    ax.imshow(im, cmap="gray"); ax.axis("off"); ax.set_title(ip.name, fontsize=9)
    for line in _lbl_of(ip).read_text().splitlines():
        if not line.strip():
            continue
        c, xc, yc, w, h = line.split()
        c = int(c); xc, yc, w, h = float(xc)*W, float(yc)*H, float(w)*W, float(h)*H
        # person은 파랑, 나머지는 빨강 — 겹쳐 그려도 구분된다
        col = "blue" if CLASS_NAMES[c] == HTP_PERSON_CLASS else "red"
        ax.add_patch(patches.Rectangle((xc-w/2, yc-h/2), w, h,
                                       fill=False, edgecolor=col, linewidth=1.2))
        ax.text(xc-w/2, max(yc-h/2-3, 0), CLASS_NAMES[c], color="white", fontsize=7,
                bbox=dict(facecolor=col, alpha=0.6, pad=0.5, edgecolor="none"))
plt.tight_layout(); plt.show()

## 4. 학습 — 범용 손그림 탐지기

COCO 사전학습 가중치에서 시작해 합성 Quick Draw 장면으로 학습한다.
산출물은 그 자체로 최종 납품 대상(**산출물 1**)이면서, `htp_finetune.ipynb`의 시작 가중치가 된다.

In [ ]:
from ultralytics import YOLO

if SKETCH_BASE.exists() and not FORCE_RETRAIN:
    print("기존 가중치 재사용:", SKETCH_BASE.resolve())
    print("  다시 학습하려면 Config에서 FORCE_RETRAIN=True (또는 SKETCH_MODEL_NAME을 새 이름으로)")
else:
    m = YOLO(MODEL_ARCH)                      # COCO 사전학습에서 시작
    m.train(data=str(qd_yaml.resolve()),
            epochs=QD_EPOCHS, imgsz=IMG_SIZE, batch=BATCH, workers=WORKERS,
            device=DEVICE, seed=SEED, project=str(RUNS_DIR), name=RUN_NAME,
            exist_ok=True, verbose=True, **STAGE_A_TRAIN_KW)
    shutil.copy(RUNS_DIR / RUN_NAME / "weights" / "best.pt", SKETCH_BASE)
    print("\n학습 완료 →", SKETCH_BASE.resolve())

## 5. 평가

범용 모델은 HTP 파인튜닝의 중간 산출물이 아니라 **독립 산출물**이므로 자체 지표를 따로 남긴다.
합성 데이터셋에는 test 분할이 없어 `val`(학습에 미사용)로 평가한다.

In [ ]:
# ── val 분할 평가 + 클래스별 지표 ─────────────────────────────────
_model = YOLO(str(SKETCH_BASE))
_r = _model.val(data=str(qd_yaml.resolve()), split="val", imgsz=IMG_SIZE,
                batch=BATCH, device=DEVICE, project=str(RUNS_DIR),
                name=f"eval_{RUN_NAME}", exist_ok=True, verbose=False)

sketch_metric = {"mAP@50": _r.box.map50, "mAP@50-95": _r.box.map,
                 "precision": _r.box.mp, "recall": _r.box.mr}
display(pd.DataFrame({SKETCH_MODEL_NAME: sketch_metric}).T.style.format("{:.4f}"))

per_cls = pd.DataFrame([
    {"클래스": CLASS_NAMES[c], "precision": _r.box.p[i], "recall": _r.box.r[i],
     "mAP@50": _r.box.ap50[i], "mAP@50-95": _r.box.ap[i]}
    for i, c in enumerate(_r.box.ap_class_index)
]).sort_values("mAP@50-95", ascending=False)
display(per_cls.style.format({c: "{:.4f}" for c in per_cls.columns if c != "클래스"}))
print("\n성능 하위 10개 클래스 (확장 시 데이터 보강 우선순위):")
display(per_cls.tail(10))

# ── HTP 스프라이트 도입 영향 확인 (S15P11B209-685) ────────────────
# person이 새로 잡히는지, 그리고 스프라이트가 하위 라벨을 함께 실어오는
# 부위 클래스들이 회귀하지 않았는지 따로 뽑아 본다. v2 수치와 비교할 것.
_watch = [HTP_PERSON_CLASS, "face", "eye", "nose", "mouth", "ear",
          "arm", "hand", "leg", "foot", "t-shirt", "shoe"]
_focus = per_cls[per_cls["클래스"].isin(_watch)]
if not _focus.empty:
    print("\n=== person + 부위 클래스 (v2 대비 회귀 확인용) ===")
    display(_focus.set_index("클래스").reindex([c for c in _watch if c in set(_focus["클래스"])])
            .style.format("{:.4f}"))

In [ ]:
# ── 실험 기록용 요약 (TSV 한 줄 → 스프레드시트) ───────────────────
import datetime

_csv = RUNS_DIR / RUN_NAME / "results.csv"
_sec = round(float(pd.read_csv(_csv).iloc[-1]["time"]), 1) if _csv.exists() else None

row = {
    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "model": SKETCH_MODEL_NAME,
    "arch": MODEL_ARCH, "imgsz": IMG_SIZE, "batch": BATCH,
    "n_classes": len(CLASS_NAMES), "per_category": QD_PER_CATEGORY,
    "train_images": QD_TRAIN_IMAGES, "epochs": QD_EPOCHS,
    "htp_person": int(bool(HTP_POOL["train"])),          # HTP 스프라이트 사용 여부
    "htp_person_prob": HTP_PERSON_PROB if HTP_POOL["train"] else 0,
    "htp_sub_labels": int(bool(HTP_SUB_LABELS and HTP_POOL["train"])),
    "train_sec": _sec,
    "mAP50": round(sketch_metric["mAP@50"], 4),
    "mAP50-95": round(sketch_metric["mAP@50-95"], 4),
    "precision": round(sketch_metric["precision"], 4),
    "recall": round(sketch_metric["recall"], 4),
}
log_row = pd.DataFrame([row])
print("=== 이번 실험 요약 (Stage A) ===")
display(log_row.T.rename(columns={0: "값"}))
print("\n=== 스프레드시트 붙여넣기용 (탭 구분) ===")
print("\t".join(log_row.columns))
print("\t".join(str(v) for v in log_row.iloc[0]))

In [ ]:
# ── 예측 결과 시각화 (val 분할) ───────────────────────────────────
val_imgs = sorted((QD_DS_DIR / "val" / "images").glob("*.jpg"))[:4]
preds = _model.predict([str(p) for p in val_imgs], imgsz=IMG_SIZE, conf=0.25,
                       device=DEVICE, verbose=False)

pred_dir = OUTPUT_ROOT / "predictions"; pred_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, len(preds), figsize=(6 * len(preds), 6))
if len(preds) == 1: axes = [axes]
for ax, p, ip in zip(axes, preds, val_imgs):
    ax.imshow(Image.open(ip).convert("RGB")); ax.axis("off"); ax.set_title(ip.stem, fontsize=8)
    b = p.boxes
    for (x1, y1, x2, y2), cls, conf in zip(b.xyxy.tolist(), b.cls.tolist(), b.conf.tolist()):
        name = CLASS_NAMES[int(cls)]
        col = "blue" if name == HTP_PERSON_CLASS else "red"
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor=col, linewidth=1.2))
        ax.text(x1, max(y1 - 3, 0), f"{name} {conf:.2f}",
                color="white", fontsize=7,
                bbox=dict(facecolor=col, alpha=0.6, pad=0.5, edgecolor="none"))
plt.tight_layout()
_out = pred_dir / f"{Path(SKETCH_MODEL_NAME).stem}_predictions.png"
fig.savefig(_out, dpi=120, bbox_inches="tight")
plt.show()
print("예측 이미지 저장:", _out.resolve())

## 정리

### 산출물
| 경로 | 내용 |
|---|---|
| `htp_yolo/<SKETCH_MODEL_NAME>` | **범용 손그림 탐지 모델 (산출물 1)** |
| `htp_yolo/stageA_quickdraw/` | 합성 데이터셋 + `data.yaml` |
| `htp_yolo/quickdraw_raw/` | Quick Draw 원본 ndjson 캐시 |
| `htp_yolo/runs/<RUN_NAME>/` | 학습 로그·곡선·`weights/best.pt` |
| `htp_yolo/runs/eval_<RUN_NAME>/` | val 평가 결과 |

### 가중치 계보 (덮어쓰지 말 것)
| 파일 | 클래스 | 비고 |
|---|---|---|
| `sketch_base.pt` | 33 | 현재 `htp_best.pt`의 시작 가중치 — 지우면 계보가 끊긴다 |
| `sketch_base_v2.pt` | 100 | Quick Draw 확장판 (person 없음) |
| `sketch_base_v3.pt` | 101 | + HTP `person`. bus가 school bus와 혼동돼 전체 mAP 하락 |
| `sketch_base_v4.pt` | 100 | v3 + `school bus`→`bus` 병합 |

### 다음 단계
HTP 전용 모델이 필요하면 [htp_finetune.ipynb](htp_finetune.ipynb)를 실행한다.
그 노트북 Config의 `SKETCH_MODEL_NAME`이 여기서 만든 파일을 가리키는지 확인할 것.

### 클래스 확장 체크리스트
1. `QD_CATEGORIES` 확장 (공식 표기 일치 확인 — 틀리면 404)
2. `QD_TRAIN_IMAGES` 상향 (Config 셀의 "클래스당 기대 인스턴스" 경고 참고)
3. `SKETCH_MODEL_NAME`·`RUN_NAME`을 새 이름으로 — 기존 가중치 보존
4. 클래스 목록이 바뀌면 합성 데이터셋은 **자동 재생성**된다.
   합성 파라미터만 바꿨다면 `htp_yolo/stageA_quickdraw/`를 직접 삭제
5. 커널 재시작 → Run All
6. `model_download.ipynb`의 `MODEL_FILES`에 새 파일 추가

### HTP person 튜닝 포인트
| 증상 | 조치 |
|---|---|
| person 인스턴스가 너무 적다/많다 | `HTP_PERSON_PROB` 조정 (분포 리포트 셀 참고) |
| `eye`·`arm` 등이 HTP 쪽으로 치우친다 | `HTP_PERSON_PROB` 하향, 또는 `HTP_SUB_LABELS=False` |
| 스프라이트에 종이 얼룩이 남는다 | `HTP_INK_FLOOR` 상향 |
| 사람이 너무 크게/작게 붙는다 | `HTP_PERSON_SCALE` 조정 |
| v2와 동일한 100클래스로 되돌리고 싶다 | `HTP_SPRITE_ENABLE=False` |